## 1. Постановка задачі

**1. Прикладна задача регресії**
Розв'язується задача прогнозування неперервної кількісної характеристики біологічного об'єкта (квітки ірису) на основі інших його просторових вимірів та біологічного виду.

**2. Визначення складових задачі:**
* **Об'єкт спостереження:** Окремий екземпляр квітки роду Ірис (*Iris*).
* **Вхідні ознаки:** Довжина чашолистка (`sepal_length`), довжина пелюстки (`petal_length`), ширина пелюстки (`petal_width`) — числові; вид квітки (`species`) — категоріальна ознака.
* **Цільова змінна:** `sepal_width` (ширина чашолистка квітки), вимірюється у сантиметрах.
* **Практичний зміст прогнозу:** Реконструкція втрачених геометричних параметрів пошкоджених гербарних зразків та біометричний контроль у селекції (виявлення аномальних пропорцій розвитку квітки).

**3. Основна експериментальна гіпотеза**
Чи дозволяють регресійні моделі машинного навчання (лінійні та нелінійні) отримати суттєво меншу похибку прогнозування ширини чашолистка порівняно з наївною baseline-моделлю (прогнозуванням середнього значення), і чи виправдане ускладнення моделі для цього набору даних?

**4. Основна метрика для порівняння**
Як основну метрику обрано **RMSE** (Root Mean Squared Error). 
* **Обґрунтування:** RMSE обчислюється і виводиться у тих самих фізичних одиницях вимірювання, що й цільова змінна (у сантиметрах). Це робить похибку легко інтерпретованою для спеціалістів предметної області (ботаніків). Крім того, математичні властивості метрики (через піднесення до квадрата перед добуванням кореня) сильніше штрафують модель за великі відхилення, що є критично важливим для уникнення грубих помилок при біометричних вимірюваннях.

## 2. Формування навчальної та тестової вибірок

Для забезпечення об'єктивного оцінювання якості моделей, набір даних розділяється на навчальну (Train) та тестову (Test) підмножини.

In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Завантаження набору даних (якщо не завантажено раніше)
iris = load_iris(as_frame=True)
df = iris.frame
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
df['species'] = df['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# Виокремлення вхідних ознак та цільової змінної
X = df.drop(columns=['sepal_width'])
y = df['sepal_width']

# Розбиття даних на навчальну та тестову вибірки (80% / 20%)
# random_state фіксується для відтворюваності результатів експерименту
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=X['species'] # Збереження балансу видів у вибірках
)

# Виведення розмірів отриманих вибірок
print(f"Розмір початкового датасету: X={X.shape}, y={y.shape}")
print(f"Розмір навчальної вибірки (Train): X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Розмір тестової вибірки (Test): X_test={X_test.shape}, y_test={y_test.shape}")

Розмір початкового датасету: X=(150, 4), y=(150,)
Розмір навчальної вибірки (Train): X_train=(120, 4), y_train=(120,)
Розмір тестової вибірки (Test): X_test=(30, 4), y_test=(30,)


**Важливе зауваження щодо тестової вибірки:**
Отримана вибірка `X_test` та `y_test` є повністю ізольованою. Вона **категорично не використовуватиметься** на етапах попередньої обробки даних (наприклад, для розрахунку середніх значень при масштабуванні), вибору архітектури моделей, налаштування гіперпараметрів чи крос-валідації. Її єдине призначення — фінальне незалежне тестування найкращої моделі наприкінці експерименту.

## 3. Перевірка preprocessing (Попередня обробка даних)

**(9) Визначення необхідних операцій попередньої обробки:**
Для набору даних Iris (де цільова змінна — `sepal_width`) необхідні такі перетворення:
*   **Заповнення пропущених значень:** Не застосовується, оскільки набір даних не містить пропусків.
*   **Масштабування числових ознак:** Необхідне. Ознаки `sepal_length`, `petal_length` та `petal_width` вимірюються в сантиметрах, але мають різний розкид значень. Для коректної роботи лінійних моделей з регуляризацією (наприклад, Ridge) застосуємо стандартизацію (`StandardScaler`).
*   **Кодування категоріальних ознак:** Необхідне. Ознаку `species` (вид ірису) потрібно перетворити на числовий формат. Використаємо `OneHotEncoder(drop='first')`, щоб уникнути мультиколінеарності (пастки фіктивних змінних).
*   **Відбір та перетворення ознак:** На базовому етапі не застосовується, оскільки набір містить лише 4 інформативні предиктори, кожен з яких є важливим для опису морфології квітки.

**(10-12) Запобігання витоку даних (Data Leakage) та використання Pipeline:**
Щоб уникнути data leakage, параметри перетворень (наприклад, середнє значення та дисперсія для `StandardScaler`) повинні розраховуватися виключно на навчальній вибірці. 
Для автоматизації цього процесу та гарантування коректної схеми `T.fit(X_train)` $\rightarrow$ `T.transform(X_test)` використаємо `ColumnTransformer` у поєднанні з `Pipeline`. Це гарантує, що під час крос-валідації передобробка буде навчатися лише на тренувальних фолдах, ізолюючи валідаційний фолд.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Визначення колонок за типами
numeric_features = ['sepal_length', 'petal_length', 'petal_width']
categorical_features = ['species']

# (11) Створення ColumnTransformer для паралельної обробки
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'
)

# Демонстрація відсутності Data Leakage: 
# На цьому етапі ми НЕ викликаємо fit_transform на всьому датасеті (X).
# Preprocessor буде інтегрований у Pipeline і навчатиметься лише всередині крос-валідації.

## 4. Побудова baseline

**(13) Створення базової моделі:**
Як baseline-модель використаємо `DummyRegressor` зі стратегією `strategy="mean"`. Ця модель ігнорує всі вхідні ознаки ($X$) і завжди прогнозує середнє значення ширини чашолистка, обчислене на навчальній вибірці. Це наївний підхід, який слугуватиме мінімальним порогом якості для наших подальших алгоритмів машинного навчання.

**(14) Оцінка baseline за допомогою Cross-Validation:**
Оцінимо базову модель за тією самою схемою 5-Fold крос-валідації, яка буде використовуватися для кандидатних моделей. Основною метрикою залишається RMSE.

In [7]:
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
import numpy as np

# (13) Створення Pipeline із базовою моделлю
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('baseline_model', DummyRegressor(strategy='mean'))
])

# (14) Оцінка baseline за схемою 5-Fold Cross-Validation на навчальній вибірці (X_train)
cv_baseline = cross_validate(
    baseline_pipeline, 
    X_train, 
    y_train, 
    cv=5, 
    scoring='neg_root_mean_squared_error',
    return_train_score=False
)

# Перетворення від'ємних значень (які scikit-learn використовує для максимізації) на стандартний RMSE
baseline_rmse_scores = -cv_baseline['test_score']
baseline_rmse_mean = np.mean(baseline_rmse_scores)
baseline_rmse_std = np.std(baseline_rmse_scores)

print("--- Результати оцінки Baseline (5-Fold CV) ---")
print(f"Середній RMSE: {baseline_rmse_mean:.4f} см")
print(f"Стандартне відхилення RMSE між fold: {baseline_rmse_std:.4f}")

--- Результати оцінки Baseline (5-Fold CV) ---
Середній RMSE: 0.4482 см
Стандартне відхилення RMSE між fold: 0.0447


## 5. Вибір кандидатних моделей

**(15-16) Обґрунтування вибору кандидатних моделей:**
Для порівняння з Baseline розглядаються три моделі з різною логікою роботи:
*   **LinearRegression (Множинна лінійна регресія):** Базовий алгоритм машинного навчання, який шукає прямі лінійні залежності між геометричними параметрами квітки та її шириною чашолистка.
*   **Ridge (Лінійна регресія з L2-регуляризацією):** Враховує можливу мультиколінеарність між розмірами пелюсток і чашолистків, додаючи штраф за занадто великі коефіцієнти, що зменшує перенавчання.
*   **RandomForestRegressor (Випадковий ліс):** Нелінійний ансамблевий алгоритм на основі дерев рішень. Обраний для перевірки наявності складних нелінійних зв'язків та взаємодій між видом ірису та розмірами його органів.

**(17) Правило вибору:** Тестова вибірка (`X_test`, `y_test`) залишається повністю ізольованою. Порівняння та вибір найкращої моделі здійснюються виключно на основі результатів крос-валідації.

---

## 6. Cross-Validation (Крос-валідація)

**(18-19) Фіксація єдиної схеми крос-валідації:**
Для об'єктивного порівняння фіксуємо єдиний об'єкт `KFold(n_splits=5, shuffle=True, random_state=42)`. Це гарантує, що базова та всі кандидатні моделі оцінюються на абсолютно однакових 5 фолдах.

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor

# (18) Фіксація єдиної схеми KFold
cv_scheme = KFold(n_splits=5, shuffle=True, random_state=42)

# (15) Визначення кандидатних моделей у поєднанні з Preprocessor
models = {
    'Baseline (Dummy)': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results_data = []

# (19-22) Оцінка всіх моделей на однакових фолдах
for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # (20) Отримання RMSE на кожному з 5 фолдів
    # scoring='neg_root_mean_squared_error' повертає від'ємне значення, тому додаємо '-'
    fold_scores = -cross_val_score(
        pipeline, 
        X_train, 
        y_train, 
        cv=cv_scheme, 
        scoring='neg_root_mean_squared_error'
    )
    
    # (21) Обчислення середнього значення та стандартного відхилення
    mean_rmse = np.mean(fold_scores)
    std_rmse = np.std(fold_scores)
    
    # (22) Форматування результату у вигляді RMSE_mean ± RMSE_std
    formatted_result = f"{mean_rmse:.4f} ± {std_rmse:.4f}"
    
    results_data.append({
        'Модель': model_name,
        'Fold 1': fold_scores[0],
        'Fold 2': fold_scores[1],
        'Fold 3': fold_scores[2],
        'Fold 4': fold_scores[3],
        'Fold 5': fold_scores[4],
        'Mean RMSE': mean_rmse,
        'Std RMSE': std_rmse,
        'Результат (RMSE_mean ± RMSE_std)': formatted_result
    })

# Формування підсумкової таблиці
df_cv_results = pd.DataFrame(results_data)
display(df_cv_results[['Модель', 'Mean RMSE', 'Std RMSE', 'Результат (RMSE_mean ± RMSE_std)']])

,Модель,Mean RMSE,Std RMSE,Результат (RMSE_mean ± RMSE_std)
0,Baseline (Dummy),0.440585,0.091673,0.4406 ± 0.0917
1,Linear Regression,0.307964,0.054351,0.3080 ± 0.0544
2,Ridge (alpha=1.0),0.318397,0.057499,0.3184 ± 0.0575
3,Random Forest,0.340851,0.042762,0.3409 ± 0.0428
